# U3-04 · Biclustering, variedades y cesta de la compra

*Machine Learning aplicado usando Python (IFCD093PO) · Unidad 3, apartados 3.4 a 3.6*

Ejecuta cada celda con **Mayús + Intro**. En el navegador t-SNE e Isomap tardan alrededor de un minuto.

> Datos (UCI, licencia CC BY 4.0), junto a este cuaderno; su preparación se explica en `LEEME.md`:
> Cardoso, M. (2013). *Wholesale customers*. https://doi.org/10.24432/C5030X ·
> Chen, D. (2012). *Online Retail II*. https://doi.org/10.24432/C5CG6D

## 1. Biclustering de clientes y productos (3.4)

In [ ]:
import numpy as np, pandas as pd
from sklearn.datasets import make_biclusters
from sklearn.cluster import SpectralCoclustering
from sklearn.metrics import consensus_score

# 1) datos sintéticos con 4 bloques ocultos y filas barajadas
D, filas, cols = make_biclusters((300, 40), 4, noise=5,
                                 shuffle=True, random_state=0)
sc = SpectralCoclustering(4, random_state=0).fit(D)
print("acierto sintético:", round(consensus_score(
    sc.biclusters_, (filas, cols)), 3))

# 2) clientes x productos: en cuántas facturas compró cada uno
t = pd.read_csv("cestas.csv")
nombres = pd.read_csv("productos.csv", index_col="producto")
A = pd.crosstab(t.cliente, t.producto)
A = A[A.sum(axis=1) >= 20]
print("matriz", A.shape)
sc = SpectralCoclustering(6, random_state=0).fit(np.log1p(A))
for b in range(6):
    f, p = sc.get_indices(b)
    dentro = A.iloc[f, p].values.mean()
    fuera = A.iloc[f].drop(columns=A.columns[p]).values.mean()
    print(b, len(f), "clientes", len(p), "productos",
          round(dentro, 2), round(fuera, 2))
    print("   ", "; ".join(nombres.descripcion[A.columns[p][:3]]))

import matplotlib.pyplot as plt
orden = A.iloc[np.argsort(sc.row_labels_),
               np.argsort(sc.column_labels_)]
plt.figure(figsize=(4.4, 2.4))
plt.imshow(np.log1p(orden.values), aspect="auto",
           interpolation="nearest")
plt.xlabel("100 productos"); plt.ylabel("1697 clientes")
plt.show()  # plt.savefig("bicluster.pdf")

**Prueba.** Cambia el número de bloques de 6 a 4 y a 8. ¿Siguen teniendo un tema reconocible?

## 2. Aprendizaje de variedades (3.5)

La primera celda prepara los datos de los mayoristas como en el apartado 3.2.

In [ ]:
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
m = pd.read_csv("mayoristas.csv")
gasto = m.drop(columns=["canal", "region"])
X = StandardScaler().fit_transform(np.log1p(gasto))

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.datasets import make_swiss_roll
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap, TSNE, trustworthiness

# 1) rollo suizo: una hoja 2D enrollada en 3D
R, t = make_swiss_roll(1500, noise=0.05, random_state=0)
metodos = {"PCA": PCA(2),
           "Isomap": Isomap(n_neighbors=10, n_components=2),
           "t-SNE": TSNE(2, perplexity=30, random_state=0)}
planos = {}
for nombre, met in metodos.items():
    E = planos[nombre] = met.fit_transform(R)
    rho = max(abs(spearmanr(E[:, j], t)[0]) for j in (0, 1))
    print(nombre, "vecinos conservados",
          round(trustworthiness(R, E, n_neighbors=10), 3),
          "orden del rollo", round(rho, 3))

# 2) perfiles de compra de los 440 mayoristas
pca = PCA().fit(X)
print("varianza explicada",
      pca.explained_variance_ratio_.cumsum().round(2))
print(pd.DataFrame(pca.components_[:2].T, index=gasto.columns,
                   columns=["CP1", "CP2"]).round(2))

import matplotlib.pyplot as plt
fig = plt.figure(figsize=(4.4, 1.3))
ax = fig.add_subplot(1, 4, 1, projection="3d")
ax.scatter(R[:, 0], R[:, 1], R[:, 2], c=t, s=1, cmap="viridis")
ax.set_axis_off(); ax.set_title("Rollo suizo")
for i, (nombre, E) in enumerate(planos.items(), start=2):
    ax = fig.add_subplot(1, 4, i)
    ax.scatter(E[:, 0], E[:, 1], c=t, s=1, cmap="viridis")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(nombre)
plt.show()  # fig.savefig("variedades.pdf")

**Prueba.** Cambia `n_neighbors=10` de Isomap por 30. ¿Sigue desplegando el rollo?

## 3. Análisis de la cesta de la compra (3.6)

In [ ]:
import numpy as np, pandas as pd

t = pd.read_csv("cestas.csv")
nombres = pd.read_csv("productos.csv", index_col="producto")
B = pd.crosstab(t.factura, t.producto).clip(upper=1)
n = len(B)
soporte = B.mean()                        # P(A)
juntos = (B.T @ B) / n                    # P(A y B)
juntos.index.name, juntos.columns.name = "si", "entonces"
reglas = juntos.stack().rename("soporte").reset_index()
reglas = reglas[reglas.si != reglas.entonces]
reglas["confianza"] = reglas.soporte / soporte[reglas.si].values
reglas["lift"] = (reglas.confianza
                  / soporte[reglas.entonces].values)
fuertes = reglas[(reglas.soporte >= .01)
                 & (reglas.confianza >= .5)]
print(n, "facturas;", len(fuertes), "reglas fuertes")
for _, r in fuertes.nlargest(6, "lift").iterrows():
    print(nombres.descripcion[r.si], "->",
          nombres.descripcion[r.entonces],
          round(r.soporte, 3), round(r.confianza, 2),
          round(r.lift, 1))
# dos superventas: ¿se compran juntos?
par = reglas[(reglas.si == "22423") & (reglas.entonces == "85123A")]
print(par.round(3).to_string(index=False))

---
## Listados de ampliación

Los listados siguientes completan los apartados del libro, en el mismo orden. Los que reutilizan un caso anterior lo importan de los ficheros `.py` que están junto a este cuaderno (por ejemplo, `from h_ventas import X, y`).

### Bloques en tablero de ajedrez  ·  `o2_ajedrez.py`

In [ ]:
import numpy as np
from sklearn.datasets import make_checkerboard
from sklearn.cluster import SpectralBiclustering, SpectralCoclustering
from sklearn.metrics import consensus_score

A, filas, cols = make_checkerboard((300, 120), n_clusters=(4, 3),
                                   noise=10, random_state=0)
rng = np.random.default_rng(0)
pf, pc = rng.permutation(300), rng.permutation(120)
B = A[pf][:, pc]                       # filas y columnas barajadas
verdad = (filas[:, pf], cols[:, pc])
for nombre, m in [
        ("ajedrez (4 x 3)", SpectralBiclustering((4, 3),
                                                 random_state=0)),
        ("co-clustering (4)", SpectralCoclustering(4,
                                                   random_state=0))]:
    m.fit(B)
    acierto = consensus_score(m.biclusters_, verdad)
    print(f"{nombre:<18} biclústeres {len(m.rows_)}"
          f"  consenso con la verdad {acierto:.3f}")

### Bloques en tablero de ajedrez  ·  `o3_ajedrez_mayoristas.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import SpectralBiclustering
from l_mayoristas import gasto, m              # 440 x 6

A = np.log1p(gasto.to_numpy())                 # gasto en logaritmo
bc = SpectralBiclustering((3, 2), method="log",
                          random_state=0).fit(A)
filas, cols = bc.row_labels_, bc.column_labels_
for g in range(2):
    print(f"categorías {g}:", ", ".join(gasto.columns[cols == g]))
nivel = pd.DataFrame([[A[filas == f][:, cols == c].mean()
                       for c in range(2)] for f in range(3)],
                     columns=["categorías 0", "categorías 1"])
nivel["clientes"] = np.bincount(filas)
nivel["% Horeca"] = [100 * (m.canal[filas == f] == 1).mean()
                     for f in range(3)]
print(nivel.round(1).to_string())

### El biplot  ·  `p4_biplot.py`

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from l_mayoristas import X, gasto, m          # 440 x 6

pca = PCA(2).fit(X)
P = pca.transform(X)
fig, eje = plt.subplots(figsize=(4.4, 3.0))
for canal, nombre in [(1, "Horeca"), (2, "minorista")]:
    s = m.canal == canal
    eje.scatter(P[s, 0], P[s, 1], s=4, alpha=0.6, label=nombre)
sube = {"frescos": 0.3, "congelados": -0.3, "drogueria": -0.3,
        "alimentacion": 0.3}                 # separar etiquetas
for j, v in enumerate(gasto.columns):        # flechas: las cargas
    x, y = 4 * pca.components_[:, j]
    eje.arrow(0, 0, x, y, color="0.2", width=0.01, head_width=0.12)
    eje.text(1.1 * x + 0.1, 1.1 * y + sube.get(v, 0), v,
             fontsize=6, ha="left", va="center",
             bbox={"fc": "white", "ec": "none", "pad": 0.5})
eje.set_xlabel("CP1"); eje.set_ylabel("CP2"); eje.legend()
plt.show()  # fig.savefig("biplot.pdf")
media = {c: P[(m.canal == c).to_numpy()].mean(axis=0).round(2)
         for c in (1, 2)}
print("CP1 y CP2 medios: Horeca", media[1], "minorista", media[2])

### Los vecinos de Isomap  ·  `p5_isomap.py`

In [ ]:
from scipy.stats import spearmanr
from sklearn.datasets import make_swiss_roll
from sklearn.manifold import Isomap

R, t = make_swiss_roll(1500, noise=0.05, random_state=0)
for k in [5, 10, 30, 100]:
    E = Isomap(n_neighbors=k, n_components=2).fit_transform(R)
    rho = max(abs(spearmanr(E[:, j], t)[0]) for j in (0, 1))
    print(f"{k:>3} vecinos: orden del rollo {rho:.3f}")

### La perplejidad de t-SNE  ·  `p2_perplejidad.py`

In [ ]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE, trustworthiness
from l_mayoristas import X, m                 # 440 mayoristas

fig, ejes = plt.subplots(1, 3, figsize=(4.4, 1.6))
for eje, perp in zip(ejes, [5, 30, 100]):
    E = TSNE(2, perplexity=perp, random_state=0).fit_transform(X)
    print(f"perplejidad {perp:>3}: vecinos conservados "
          f"{trustworthiness(X, E, n_neighbors=10):.3f}")
    eje.scatter(E[:, 0], E[:, 1], c=m.canal, s=2, cmap="coolwarm")
    eje.set_title(f"perplejidad {perp}")
    eje.set_axis_off()
plt.show()  # fig.savefig("perplejidad.pdf")

### Comprimir con PCA antes de agrupar  ·  `p3_comprimir.py`

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from l_mayoristas import X                    # 440 x 6

base = KMeans(3, n_init=10, random_state=0).fit_predict(X)
for k in range(1, 7):
    pca = PCA(k).fit(X)
    Xr = pca.inverse_transform(pca.transform(X))
    error = np.mean((X - Xr) ** 2)
    e = KMeans(3, n_init=10, random_state=0).fit_predict(
        pca.transform(X))
    ari = adjusted_rand_score(base, e)
    print(f"{k} comp.: varianza "
          f"{pca.explained_variance_ratio_.sum():4.0%}, error "
          f"{error:.3f}, ARI con los 6 originales {ari:.3f}")

### Más de dos artículos: el principio Apriori  ·  `q2_tripletas.py`

In [ ]:
from itertools import combinations
import pandas as pd
t = pd.read_csv("cestas.csv", dtype=str)
nombres = pd.read_csv("productos.csv", dtype=str,
                      index_col="producto").descripcion
cestas = t.groupby("factura").producto.agg(frozenset)
n, minimo = len(cestas), 0.01                # soporte mínimo 1 %

def frecuentes(candidatos):
    """Cuenta cada candidato y se queda con los frecuentes."""
    cuenta = {c: sum(c <= b for b in cestas) for c in candidatos}
    return {c: v / n for c, v in cuenta.items() if v / n >= minimo}

nivel = frecuentes({frozenset([p]) for p in t.producto.unique()})
todos = dict(nivel)
for k in (2, 3):
    previos = list(nivel)
    cand = {a | b for a, b in combinations(previos, 2)
            if len(a | b) == k}
    # Apriori: todo subconjunto de un frecuente es frecuente
    cand = {c for c in cand
            if all(c - {x} in nivel for x in c)}
    nivel = frecuentes(cand)
    todos.update(nivel)
    print(f"tamaño {k}: {len(cand)} candidatos, "
          f"{len(nivel)} frecuentes")
for c, s in sorted(nivel.items(), key=lambda x: -x[1])[:3]:
    print(f"{s:.3f}", " + ".join(nombres[p][:18] for p in sorted(c)))

### Cuántas reglas salen según los umbrales  ·  `q4_umbrales.py`

In [ ]:
import pandas as pd
t = pd.read_csv("cestas.csv", dtype=str)
B = pd.crosstab(t.factura, t.producto).clip(upper=1)
n, p = len(B), B.mean()
juntos = (B.T @ B) / n
r = juntos.stack().rename("soporte").rename_axis(
    ["si", "entonces"]).reset_index()
r = r[r.si != r.entonces]
r["confianza"] = r.soporte / p[r.si].values
r["lift"] = r.confianza / p[r.entonces].values
tabla = pd.DataFrame({f"conf. {c:.0%}": [
    ((r.soporte >= s) & (r.confianza >= c)).sum()
    for s in (0.005, 0.01, 0.02, 0.03)] for c in (0.2, 0.4, 0.6)},
    index=["soporte 0,5 %", "soporte 1 %", "soporte 2 %",
           "soporte 3 %"])
print(tabla.to_string())
print(f"pares con lift < 1: {(r.lift < 1).mean():.0%}"
      f" de {len(r)}")

### Productos parecidos  ·  `q5_similares.py`

In [ ]:
import numpy as np
import pandas as pd
t = pd.read_csv("cestas.csv", dtype=str)
nombres = pd.read_csv("productos.csv", dtype=str,
                      index_col="producto").descripcion
B = pd.crosstab(t.factura, t.producto).clip(upper=1)
juntos = B.T @ B
n = np.diag(juntos)
jac = juntos / (n[:, None] + n[None, :] - juntos)   # Jaccard
jac = jac.mask(np.eye(len(jac), dtype=bool), 0)

def parecidos(codigo, k=3):
    s = jac[codigo].nlargest(k)
    return [(nombres[p][:28], round(v, 2)) for p, v in s.items()]

for codigo in ["85123A", "22423", "20725"]:
    print(nombres[codigo][:30])
    for nombre, v in parecidos(codigo):
        print(f"   {v:.2f}  {nombre}")

### Qué compran los mejores clientes  ·  `q6_segmentos.py`

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from m_rfm import c, Z                     # 5878 clientes RFM

t = pd.read_csv("cestas.csv", dtype={"producto": str})
nombres = pd.read_csv("productos.csv", dtype=str,
                      index_col="producto").descripcion
seg = pd.Series(KMeans(4, n_init=10, random_state=0)
                .fit_predict(Z), index=c.index)
mejores = c.groupby(seg).frecuencia.median().idxmax()
t["mejores"] = t.cliente.map(seg).eq(mejores)
facturas = t.groupby("mejores").factura.nunique()
cuota = (t.groupby(["mejores", "producto"]).factura.nunique()
         .div(facturas, level=0).unstack(0))
cuota["razon"] = cuota[True] / cuota[False]
print("facturas: mejores clientes", facturas[True],
      "| resto", facturas[False])
for nombre, fila in [("más propios de los mejores",
                      cuota.nlargest(3, "razon")),
                     ("más propios del resto",
                      cuota.nsmallest(3, "razon"))]:
    print(nombre + ":")
    for p, r in fila.razon.items():
        print(f"   {r:4.2f}  {nombres[p][:34]}")

### Confianza y lift a la vez  ·  `q7_mapa_reglas.py`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
t = pd.read_csv("cestas.csv", dtype=str)
B = pd.crosstab(t.factura, t.producto).clip(upper=1)
n, p = len(B), B.mean()
j = (B.T @ B / n).to_numpy(copy=True)
np.fill_diagonal(j, np.nan)
conf = j / p.to_numpy()[:, None]               # P(B | A), fila A
lift = conf / p.to_numpy()[None, :]
ok = ~np.isnan(j)
s, c, l = j[ok], conf[ok], lift[ok]
print(f"{ok.sum()} reglas A -> B")
print("correlación de rangos confianza-lift:",
      round(pd.Series(c).corr(pd.Series(l), method="spearman"), 2))
for umbral in [2, 5, 10]:
    print(f"lift > {umbral:>2}: {(l > umbral).sum():4d} reglas, "
          f"confianza mediana {np.median(c[l > umbral]):.2f}")
fig, eje = plt.subplots(figsize=(4.4, 2.4))
pts = eje.scatter(c, l, c=s, s=2, cmap="viridis", vmax=0.02)
eje.set_yscale("log"); eje.set_xlabel("confianza")
eje.set_ylabel("lift (escala log.)")
fig.colorbar(pts, label="soporte"); plt.show()  # fig.savefig("reglas.pdf")

### De las reglas a la recomendación  ·  `q3_recomendar.py`

In [ ]:
import numpy as np
import pandas as pd
t = pd.read_csv("cestas.csv", dtype=str)
B = pd.crosstab(t.factura, t.producto).clip(upper=1)
corte = int(len(B) * 0.8)                # facturas en orden de número
ent, pru = B.iloc[:corte], B.iloc[corte:]
juntos = ent.T @ ent                     # facturas con A y con B
confianza = juntos.div(np.diag(juntos), axis=0)   # P(B | A)
confianza = confianza.mask(np.eye(len(juntos), dtype=bool), 0)
populares = ent.sum().sort_values(ascending=False).index

rng = np.random.default_rng(0)
aciertos = {"reglas": 0, "más vendidos": 0}
casos = 0
for _, fila in pru.iterrows():
    cesta = list(fila.index[fila == 1])
    if len(cesta) < 2:
        continue
    oculto = rng.choice(cesta)            # el producto a adivinar
    resto = [p for p in cesta if p != oculto]
    punt = confianza.loc[resto].max().drop(resto)
    top = {"reglas": punt.nlargest(5).index,
           "más vendidos": [p for p in populares
                            if p not in resto][:5]}
    for k in aciertos:
        aciertos[k] += oculto in top[k]
    casos += 1
print(f"{casos} cestas de prueba con 2 o más productos")
for k, v in aciertos.items():
    print(f"{k:<13} acierta entre 5 propuestas: {v / casos:.1%}")

**Ejercicio.** Muestra las cinco reglas con mayor confianza cuyo consecuente sea `85123A`. ¿Tienen todas un lift alto?

> Agrawal, R., Imieliński, T. y Swami, A. (1993). Mining association rules between sets of items in large databases. *Proceedings of the 1993 ACM SIGMOD*, 207–216. https://doi.org/10.1145/170035.170072